# Optimizer sandbox

Play with a real `DataPackage` outside of Frappe/bench: tweak employees, FTE targets, rules and weights, re-solve, and inspect the result. No live site needed once you have a snapshot.

**1. Capture a snapshot** from an existing Optimizer Run on `development.localhost` (any status — only its `date`/`mode`/`ruleset`/leave-speculations/existing-assignments mode are used):

```bash
bench --site development.localhost capture-datapackage --run <run-name>
```

This writes `sandbox/snapshots/<run-name>.json`. Snapshots contain real employee/leave data from your dev site, so `sandbox/snapshots/` is gitignored — re-capture instead of committing one.

**2. Pick a kernel**: the app's own Python env (`uv run python` from `apps/autoshift`, or the bench env — both already have `pulp`, `pandas`, and `autoshift` importable; see `pyproject.toml`'s `dev` dependency group for `ipykernel`/`pandas`).

In [1]:
import logging
from pathlib import Path

import pulp
from helpers import *

from autoshift.optimizer.rules import BUILTIN_RULES
from autoshift.optimizer.types import DataPackage

logging.basicConfig(level=logging.DEBUG)
logging.getLogger().setLevel(logging.DEBUG)

## Load a snapshot

Point this at whichever file `capture-datapackage` produced.

In [2]:
SNAPSHOT = Path("snapshots/AS-2026-06-22-052.json")

data = DataPackage.loads(SNAPSHOT.read_text())
active_rule_count = len(data.rules) or len(BUILTIN_RULES)
print(
	f"{len(data.employees)} employees, {len(data.shift_types)} shift types, "
	f"{len(data.working_days)} days ({data.working_days[0]}..{data.working_days[-1]}), "
	f"{len(data.branches)} branches, {active_rule_count} rules"
)

54 employees, 2 shift types, 20 days (2026-06-22..2026-07-17), 1 branches, 8 rules


## Solve as captured

In [3]:
prob, x, active_rooms, logs = solve(data)
print(status(prob), pulp.value(prob.objective))

DEBUG:pulp.apis.core:cbc /tmp/7f33bb4c9a8c457f96c3444711fee8f9-pulp.mps -max -sec 30 -timeMode elapsed -solve -printingOptions all -solution /tmp/7f33bb4c9a8c457f96c3444711fee8f9-pulp.sol 


Optimal 119.765625


In [4]:
prob.constraints()[0].name

'one_branch:one_branch_191_Omni_AM_2026_06_22'

## Inspect the solution

In [ ]:
ass = assignment_frame(data, x)
ass

,employee,shift_type,date,branch,forced,assigned
0,120,Omni AM,2026-06-22,Balexert,False,1.0
1,157,Omni AM,2026-06-22,Balexert,False,1.0
2,175,Omni AM,2026-06-22,Balexert,False,1.0
3,181,Omni AM,2026-06-22,Balexert,False,1.0
4,187,Omni AM,2026-06-22,Balexert,False,1.0
...,...,...,...,...,...,...
235,138,Omni PM,2026-07-17,Balexert,False,1.0
236,166,Omni PM,2026-07-17,Balexert,False,1.0
237,177,Omni PM,2026-07-17,Balexert,False,1.0
238,187,Omni PM,2026-07-17,Balexert,False,1.0


In [6]:
constraint_frame(prob)

,len,type,constraint,slack,pi
0,38,one_branch,x_1_Omni_AM_2026_06_22_Balexert <= 1.0,1.0,-0.0
1,38,one_branch,x_1_Omni_AM_2026_06_23_Balexert <= 1.0,1.0,-0.0
2,38,one_branch,x_1_Omni_AM_2026_06_24_Balexert <= 1.0,1.0,-0.0
3,38,one_branch,x_1_Omni_AM_2026_06_25_Balexert <= 1.0,1.0,-0.0
4,38,one_branch,x_1_Omni_AM_2026_06_26_Balexert <= 1.0,1.0,-0.0
...,...,...,...,...,...
3275,552,room_coverage,-ar_Omni___CMB&B_Omni_PM_2026_07_13_Balexert +...,-0.0,-0.0
3276,552,room_coverage,-ar_Omni___CMB&B_Omni_PM_2026_07_14_Balexert +...,-0.0,-0.0
3277,552,room_coverage,-ar_Omni___CMB&B_Omni_PM_2026_07_15_Balexert +...,-0.0,-0.0
3278,552,room_coverage,-ar_Omni___CMB&B_Omni_PM_2026_07_16_Balexert +...,-0.0,-0.0


In [ ]:
ass[ass["employee"] == "113"]

,employee,shift_type,date,branch,forced,assigned
7,113,Omni PM,2026-06-22,Balexert,False,1.0
25,113,Omni AM,2026-06-24,Balexert,False,1.0
43,113,Omni PM,2026-06-25,Balexert,False,1.0
49,113,Omni AM,2026-06-26,Balexert,False,1.0
67,113,Omni PM,2026-06-29,Balexert,False,1.0
73,113,Omni AM,2026-06-30,Balexert,False,1.0
84,113,Omni AM,2026-07-01,Balexert,False,1.0
97,113,Omni AM,2026-07-02,Balexert,False,1.0
108,113,Omni AM,2026-07-03,Balexert,False,1.0
127,113,Omni PM,2026-07-06,Balexert,False,1.0


In [8]:
room_utilization_frame(data, active_rooms)

,discipline,shift_type,date,branch,staffed,capacity
0,Omni - CMB&B,Omni AM,2026-06-22,Balexert,6,6
1,Omni - CMB&B,Omni PM,2026-06-22,Balexert,6,6
2,Omni - CMB&B,Omni AM,2026-06-23,Balexert,6,6
3,Omni - CMB&B,Omni PM,2026-06-23,Balexert,6,6
4,Omni - CMB&B,Omni AM,2026-06-24,Balexert,6,6
5,Omni - CMB&B,Omni PM,2026-06-24,Balexert,6,6
6,Omni - CMB&B,Omni AM,2026-06-25,Balexert,6,6
7,Omni - CMB&B,Omni PM,2026-06-25,Balexert,6,6
8,Omni - CMB&B,Omni AM,2026-06-26,Balexert,6,6
9,Omni - CMB&B,Omni PM,2026-06-26,Balexert,6,6


In [9]:
# per-rule contribution to the objective (built-in objective rules only — see helpers.py)
objective_breakdown(data, x, active_rooms)

{'Objective: Conserve Existing Assignments': -0.234375,
 'Objective: Room utilization': 240.0,
 'Objective: Shift preferences': -120.0}

## Schedule grid

Simplified, offline view of the solved schedule (employee x day), analogous to
`OptimizerRun.get_schedule_events` but without the Frappe/DB-backed "existing" overlay.

In [10]:
from helpers import schedule_grid  # ty:ignore[unresolved-import]

schedule_grid(data, x)

,2026-06-22,2026-06-23,2026-06-24,2026-06-25,2026-06-26,2026-06-29,2026-06-30,2026-07-01,2026-07-02,2026-07-03,2026-07-06,2026-07-07,2026-07-08,2026-07-09,2026-07-10,2026-07-13,2026-07-14,2026-07-15,2026-07-16,2026-07-17
1,,,,,,,,,,,,,,,,,,,,
10,,,,,,,,,,,,,,,,,,,,
103,1@0,0@0,0@0,1@0,0@0,1@0,0@0,,0@0,,1@0,0@0,,1@0,0@0,1@0,0@0,1@0,,1@0
105,,,,,,,,,,,,,,,,,,,,
106,,,,,,,,,,,,,,,,,,,,
108,,,,,,,,,,,,,,,,,,,,
111,,,,,,,,,,,,,,,,,,,,
113,1@0,,0@0,1@0,0@0,1@0,0@0,0@0,0@0,0@0,1@0,1@0,0@0,,0@0,0@0,1@0,0@0,0@0,
116,,,,,,,,,,,,,,,,,,,,
119,,,,,,,,,,,,,,,,,,,,


## Play with variables

`DataPackage` is frozen — use `replace(data, field=...)` (shorthand for `dataclasses.replace`) to build a variant, then re-solve and compare.

In [35]:
variant = data  # start here, then override fields above
prob2, x2, active_rooms2, _ = built = solve(variant, solve=False)

In [36]:
for var in prob2.variables():
	var.cat = pulp.LpContinuous

In [37]:
print(status(prob2), pulp.value(prob2.objective))
prob2, x2, active_rooms2, _ = built = solve(variant, solve=built)
print(status(prob2), pulp.value(prob2.objective))
assignment_frame(variant, x2)

DEBUG:pulp.apis.core:cbc /tmp/5a85e3a131b54e1b95fc88e5682129a3-pulp.mps -max -sec 30 -timeMode elapsed -solve -printingOptions all -solution /tmp/5a85e3a131b54e1b95fc88e5682129a3-pulp.sol 


Not Solved None
Optimal 119.765625


,employee,shift_type,date,branch,forced,assigned
0,120,Omni AM,2026-06-22,Balexert,False,1.0
1,157,Omni AM,2026-06-22,Balexert,False,1.0
2,175,Omni AM,2026-06-22,Balexert,False,1.0
3,181,Omni AM,2026-06-22,Balexert,False,1.0
4,187,Omni AM,2026-06-22,Balexert,False,1.0
...,...,...,...,...,...,...
235,138,Omni PM,2026-07-17,Balexert,False,1.0
236,166,Omni PM,2026-07-17,Balexert,False,1.0
237,177,Omni PM,2026-07-17,Balexert,False,1.0
238,187,Omni PM,2026-07-17,Balexert,False,1.0


In [38]:
(a := constraint_frame(prob2))

,len,type,constraint,slack,pi
0,38,one_branch,x_1_Omni_AM_2026_06_22_Balexert <= 1.0,1.0,-0.000000
1,38,one_branch,x_1_Omni_AM_2026_06_23_Balexert <= 1.0,1.0,-0.000000
2,38,one_branch,x_1_Omni_AM_2026_06_24_Balexert <= 1.0,1.0,-0.000000
3,38,one_branch,x_1_Omni_AM_2026_06_25_Balexert <= 1.0,1.0,-0.000000
4,38,one_branch,x_1_Omni_AM_2026_06_26_Balexert <= 1.0,1.0,-0.000000
...,...,...,...,...,...
3275,552,room_coverage,-ar_Omni___CMB&B_Omni_PM_2026_07_13_Balexert +...,-0.0,-0.750977
3276,552,room_coverage,-ar_Omni___CMB&B_Omni_PM_2026_07_14_Balexert +...,-0.0,-0.750977
3277,552,room_coverage,-ar_Omni___CMB&B_Omni_PM_2026_07_15_Balexert +...,-0.0,-0.750977
3278,552,room_coverage,-ar_Omni___CMB&B_Omni_PM_2026_07_16_Balexert +...,-0.0,-0.750977


In [39]:
a.nunique()

len              7
type             3
constraint    3280
slack            2
pi               3
dtype: int64

In [ ]:
a["slack"].value_counts()

slack
 1.0    2760
-0.0     520
Name: count, dtype: int64

In [ ]:
a["type"].value_counts()

type
one_branch       2160
one_shift        1080
room_coverage      40
Name: count, dtype: int64

In [ ]:
a["len"].value_counts().sort_index()

len
38       80
39      680
40     1400
72       40
74      340
76      700
552      40
Name: count, dtype: int64

In [43]:
a[
	[
		"type",
		"slack",
		"pi",
	]
].value_counts().sort_index()

type           slack  pi       
one_branch     -0.0   -0.000000     240
                1.0   -0.000000    1920
one_shift      -0.0   -0.000000     240
                1.0   -0.000000     840
room_coverage  -0.0   -0.750977      20
                      -0.250977      20
Name: count, dtype: int64

In [ ]:
a.drop_duplicates(subset=["type", "pi", "slack"])

,len,type,constraint,slack,pi
0,38,one_branch,x_1_Omni_AM_2026_06_22_Balexert <= 1.0,1.0,-0.0
80,39,one_branch,x_23_Omni_AM_2026_06_23_Balexert <= 1.0,-0.0,-0.0
2120,40,thefirstrule,x_191_Omni_AM_2026_06_22_Balexert <= 1.0,1.0,-0.0
2160,72,one_shift,x_1_Omni_AM_2026_06_22_Balexert + x_1_Omni_PM_...,1.0,-0.0
2200,74,one_shift,x_23_Omni_AM_2026_06_23_Balexert + x_23_Omni_P...,-0.0,-0.0
3240,552,room_coverage,-ar_Omni___CMB&B_Omni_AM_2026_06_22_Balexert +...,-0.0,-0.0


In [ ]:
variant3 = data  # start here, then override fields above
prob3, x3, active_rooms3, _ = built = solve(variant, solve=False)

DEBUG:pulp.apis.core:cbc /tmp/49be4fde60334fc5bb6c048336754f7b-pulp.mps -max -sec 30 -timeMode elapsed -solve -printingOptions all -solution /tmp/49be4fde60334fc5bb6c048336754f7b-pulp.sol 


Not Solved None
Optimal 119.765625


type            pi    slack
one_branch      -0.0  -0.0      240
                       1.0     1919
one_shift       -0.0  -0.0      240
                       1.0      840
room_coverage   -0.0  -0.0       40
thefirstrule||  -0.0   1.0        1
Name: count, dtype: int64

In [ ]:
prob3.constraints()[0].name = "thefirstrule||:"

print(status(prob3), pulp.value(prob3.objective))
prob3, x3, active_rooms3, _ = built = solve(variant, solve=built)
print(status(prob3), pulp.value(prob3.objective))
# assignment_frame(variant, x3)
(a := constraint_frame(prob3))
# a[['type', 'pi', 'slack']].value_counts().sort_index()
a.drop_duplicates(subset=["type", "pi", "slack"])

DEBUG:pulp.apis.core:cbc /tmp/1fa89969487e4446ab6d1b39a937579f-pulp.mps -max -sec 30 -timeMode elapsed -solve -printingOptions all -solution /tmp/1fa89969487e4446ab6d1b39a937579f-pulp.sol 


Optimal 119.765625
Optimal 119.765625


,len,type,constraint,slack,pi
0,38,one_branch,x_1_Omni_AM_2026_06_22_Balexert <= 1.0,1.0,-0.0
80,39,one_branch,x_23_Omni_AM_2026_06_23_Balexert <= 1.0,-0.0,-0.0
2120,40,thefirstrule||,x_191_Omni_AM_2026_06_22_Balexert <= 1.0,1.0,-0.0
2160,72,one_shift,x_1_Omni_AM_2026_06_22_Balexert + x_1_Omni_PM_...,1.0,-0.0
2200,74,one_shift,x_23_Omni_AM_2026_06_23_Balexert + x_23_Omni_P...,-0.0,-0.0
3240,552,room_coverage,-ar_Omni___CMB&B_Omni_AM_2026_06_22_Balexert +...,-0.0,-0.0


In [ ]:
# example: give one employee a bigger FTE target
# employee = data.employees[0]
# variant = replace(data, target_shifts={**data.target_shifts, employee: 40})

variant = data  # start here, then override fields above
prob4, x4, active_rooms4, _ = solve(variant)
print(status(prob4), pulp.value(prob4.objective))
assignment_frame(variant, x4)

DEBUG:pulp.apis.core:cbc /tmp/70a6d49927bb440590268b3350452bcb-pulp.mps -max -sec 30 -timeMode elapsed -solve -printingOptions all -solution /tmp/70a6d49927bb440590268b3350452bcb-pulp.sol 


Optimal 119.765625


,employee,shift_type,date,branch,forced,assigned
0,120,Omni AM,2026-06-22,Balexert,False,1.0
1,157,Omni AM,2026-06-22,Balexert,False,1.0
2,175,Omni AM,2026-06-22,Balexert,False,1.0
3,181,Omni AM,2026-06-22,Balexert,False,1.0
4,187,Omni AM,2026-06-22,Balexert,False,1.0
...,...,...,...,...,...,...
235,138,Omni PM,2026-07-17,Balexert,False,1.0
236,166,Omni PM,2026-07-17,Balexert,False,1.0
237,177,Omni PM,2026-07-17,Balexert,False,1.0
238,187,Omni PM,2026-07-17,Balexert,False,1.0


## Tweak rule weights / selection

`data.rules` is a tuple of `(rule_name, builtin_key, custom_code, weight)`. An empty tuple means "every built-in rule at weight 1.0". Rebuild it to change weights, drop constraint rules (careful — most exist for correctness, not just "nice to have"), or add a rule you're drafting in `autoshift/optimizer/rule_scratchpad.py`.

In [22]:
# # example: double the room-utilization objective's weight
# base_rules = data.rules or tuple((rule.title, k, "", 1.0) for k, rule in BUILTIN_RULES.items())
# reweighted = replace(
# 	data,
# 	rules=tuple(
# 		(name, key, code, weight * 2 if key == "room_utilization_objective" else weight)
# 		for name, key, code, weight in base_rules
# 	),
# )
# prob3, x3, active_rooms3, _ = solve(reweighted)
# objective_breakdown(reweighted, x3, active_rooms3)

In [23]:
# example: double the room-utilization objective's weight
import itertools

from autoshift.optimizer.rules import RuleContext

base_rules = data.rules or tuple((rule.title, k, "", 1.0) for k, rule in BUILTIN_RULES.items())


def weigh_assignments_objective(ctx: RuleContext) -> None:
	data = ctx.data
	epsilon = 2**-10
	ctx.add_objective(
		pulp.lpSum((0 if comb in data.forced else -epsilon) * var for comb, var in ctx.x.items())
	)


new_rule = (
	"Objective: Conserve Existing Assignments",
	"",
	"""
def apply(ctx: RuleContext) -> None:
	data = ctx.data
	epsilon = 2**-10
	ctx.add_objective(
		pulp.lpSum((0 if comb in data.forced else -epsilon) * var for comb, var in ctx.x.items())
	)
	""",
	1.0,
)
reweighted = replace(
	data,
	rules=[*base_rules, new_rule],
)
prob3, x3, active_rooms3, _ = solve(reweighted)
objective_breakdown(reweighted, x3, active_rooms3)

DEBUG:pulp.apis.core:cbc /tmp/fe2ab18efa5f4b718dbb550777b99f8e-pulp.mps -max -sec 30 -timeMode elapsed -solve -printingOptions all -solution /tmp/fe2ab18efa5f4b718dbb550777b99f8e-pulp.sol 


{'Objective: Conserve Existing Assignments': -0.234375,
 'Objective: Room utilization': 240.0,
 'Objective: Shift preferences': -120.0}

In [24]:
df = assignment_frame(reweighted, x3)

In [25]:
df.nunique()

employee      14
shift_type     2
date          20
branch         1
forced         1
assigned       1
dtype: int64

In [147]:
p = pulp.LpProblem("helelo", pulp.LpMaximize)
x = p.add_variable("x", 0, 5)
y = p.add_variable("y", 0, 10)
p += x + y
# p += x <= 5  # upbound
# p += y <= 10  # upbound
p += x + y <= 12
p.solve(pulp.PULP_CBC_CMD(msg=False))

print(f"Status: {pulp.LpStatus[p.status]}")
print(f"Optimal x: {pulp.value(x)}, y: {pulp.value(y)}")
print(f"Objective: {pulp.value(p.objective)}")
for constraint_name, constraint in p.constraints.items():
	pi = constraint.pi  # Dual value / shadow price
	print(f"{constraint_name}: slack={constraint.slack:.2f}, pi={pi}")
import math

obj_value = pulp.value(p.objective)

p += p.objective == obj_value - 2e-20

solutions = [
	{
		v.name: [
			v.varValue,
			v.dj,
			v.varValue - v.upBound,
		]
		for v in p.variables()
	}
]
while not next(
	(
		setattr(v, "upBound", v.upBound - 0.125)
		for v in p.variables()
		if math.isclose(v.dj, (v.varValue - v.upBound), abs_tol=2e-20)
		and math.isclose((v.varValue - v.upBound), 0.0, abs_tol=2e-20)
	),
	True,
):
	p.solve()
	solutions.append(
		{
			**{
				v.name: [
					v.varValue,
					v.dj,
					v.varValue - v.upBound,
				]
				for v in p.variables()
			},
			**{
				f"c_{c.name}": [
					c.pi,
					c.slack,
				]
				for c in p.constraints()
			},
		}
	)
	if p.status != pulp.LpStatusOptimal:
		print(f"not opti {p.status=}")
		break
	if pulp.value(p.objective) < obj_value:
		print(f"objectif lune {pulp.value(p.objective)} < {obj_value}")
		break
solutions

DEBUG:pulp.apis.core:/workspace/development/frappe-bench/apps/autoshift/.venv/lib/python3.14/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/89c0b9d41ed24ff1999c4bb0becd25d2-pulp.mps -max -timeMode elapsed -solve -printingOptions all -solution /tmp/89c0b9d41ed24ff1999c4bb0becd25d2-pulp.sol 
DEBUG:pulp.apis.core:/workspace/development/frappe-bench/apps/autoshift/.venv/lib/python3.14/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/6b4d77e0b11242e59b441920f1a8aa77-pulp.mps -max -timeMode elapsed -solve -printingOptions all -solution /tmp/6b4d77e0b11242e59b441920f1a8aa77-pulp.sol 
DEBUG:pulp.apis.core:/workspace/development/frappe-bench/apps/autoshift/.venv/lib/python3.14/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/e1214073106a4e4b8c8bf65219f6d7f8-pulp.mps -max -timeMode elapsed -solve -printingOptions all -solution /tmp/e1214073106a4e4b8c8bf65219f6d7f8-pulp.sol 


DEBUG:pulp.apis.core:/workspace/development/frappe-bench/apps/autoshift/.venv/lib/python3.14/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/b73987a1defc4e9b93333777574f2da2-pulp.mps -max -timeMode elapsed -solve -printingOptions all -solution /tmp/b73987a1defc4e9b93333777574f2da2-pulp.sol 
DEBUG:pulp.apis.core:/workspace/development/frappe-bench/apps/autoshift/.venv/lib/python3.14/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/203f5126b4354e349d7890c61ec7e89d-pulp.mps -max -timeMode elapsed -solve -printingOptions all -solution /tmp/203f5126b4354e349d7890c61ec7e89d-pulp.sol 
DEBUG:pulp.apis.core:/workspace/development/frappe-bench/apps/autoshift/.venv/lib/python3.14/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/9e785307d4224770b91bb3dd35b308c8-pulp.mps -max -timeMode elapsed -solve -printingOptions all -solution /tmp/9e785307d4224770b91bb3dd35b308c8-pulp.sol 
DEBUG:pulp.apis.core:/workspace/development/frappe-bench/apps/autoshift/.venv/lib/python

Status: Optimal
Optimal x: 2.0, y: 10.0
Objective: 12.0
_C1: slack=-0.00, pi=1.0


DEBUG:pulp.apis.core:/workspace/development/frappe-bench/apps/autoshift/.venv/lib/python3.14/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/3b34c2e7f5ab4e2aa098022bb770d891-pulp.mps -max -timeMode elapsed -solve -printingOptions all -solution /tmp/3b34c2e7f5ab4e2aa098022bb770d891-pulp.sol 
DEBUG:pulp.apis.core:/workspace/development/frappe-bench/apps/autoshift/.venv/lib/python3.14/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/a43a878c303c43e7becb09e3be2cbca0-pulp.mps -max -timeMode elapsed -solve -printingOptions all -solution /tmp/a43a878c303c43e7becb09e3be2cbca0-pulp.sol 
DEBUG:pulp.apis.core:/workspace/development/frappe-bench/apps/autoshift/.venv/lib/python3.14/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/ac0caf3dd1c242358d11e84c38a0b554-pulp.mps -max -timeMode elapsed -solve -printingOptions all -solution /tmp/ac0caf3dd1c242358d11e84c38a0b554-pulp.sol 
DEBUG:pulp.apis.core:/workspace/development/frappe-bench/apps/autoshift/.venv/lib/python

[{'x': [2.0, 0.0, -3.0], 'y': [10.0, 0.0, 0.0]},
 {'x': [2.125, 0.0, -2.875], 'y': [9.875, 0.0, 0.0], 'c_None': [1.0, -0.0]},
 {'x': [2.25, 0.0, -2.75], 'y': [9.75, 0.0, 0.0], 'c_None': [1.0, -0.0]},
 {'x': [2.375, 0.0, -2.625], 'y': [9.625, 0.0, 0.0], 'c_None': [1.0, -0.0]},
 {'x': [2.5, 0.0, -2.5], 'y': [9.5, 0.0, 0.0], 'c_None': [1.0, -0.0]},
 {'x': [2.625, 0.0, -2.375], 'y': [9.375, 0.0, 0.0], 'c_None': [1.0, -0.0]},
 {'x': [2.75, 0.0, -2.25], 'y': [9.25, 0.0, 0.0], 'c_None': [1.0, -0.0]},
 {'x': [2.875, 0.0, -2.125], 'y': [9.125, 0.0, 0.0], 'c_None': [1.0, -0.0]},
 {'x': [3.0, 0.0, -2.0], 'y': [9.0, 0.0, 0.0], 'c_None': [1.0, -0.0]},
 {'x': [3.125, 0.0, -1.875], 'y': [8.875, 0.0, 0.0], 'c_None': [1.0, -0.0]},
 {'x': [3.25, 0.0, -1.75], 'y': [8.75, 0.0, 0.0], 'c_None': [1.0, -0.0]},
 {'x': [3.375, 0.0, -1.625], 'y': [8.625, 0.0, 0.0], 'c_None': [1.0, -0.0]},
 {'x': [3.5, 0.0, -1.5], 'y': [8.5, 0.0, 0.0], 'c_None': [1.0, -0.0]},
 {'x': [3.625, 0.0, -1.375], 'y': [8.375, 0.0, 0.0], '

In [141]:
{
	v.name: [
		v.varValue,
		v.dj,
		v.varValue - v.upBound,
	]
	for v in p.variables()
}

{'x': [2.3, 0.0, -2.7], 'y': [9.7, 0.0, -1.7763568394002505e-15]}

In [143]:
p

helelo:
MAXIMIZE
1*x + 1*y + 0
SUBJECT TO
_C1: x + y <= 12

_C2: x + y = 12

VARIABLES
x <= 5 Continuous
y <= 9.7 Continuous